# Final Hackathon Model and Results

This notebook shows the final approach used for the VectorMesh hackathon. The task is multi-label classification of Dutch legal documents into legal fact labels. I use an attention model as the main classifier, improve rare-label behaviour with cost-sensitive training and per-label thresholds, and add a small legal phrase router for confusing `art. 7:3` labels.

## Setup

The training code is included below. `RUN_TRAINING` is set to `False` by default so the notebook opens quickly and shows the saved results. Set it to `True` to rerun the final model training from the cached embeddings.

In [ ]:
from pathlib import Path
import random
import re

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from vectormesh.components import AttentionAggregator, FixedPadding, NeuralNet, Serial
from vectormesh.data import Collate, OneHot
from vectormesh.data.cache import VectorCache
from vectormesh.data.vectorizers import detect_device

RUN_TRAINING = False
SEED = 42
N_CLASSES = 32
EMBEDDING_COL = "legal_dutch"

MAX_CHUNKS = 40
BATCH_SIZE = 32
EPOCHS = 8
LR = 1e-3
POS_WEIGHT_CAP = 4.0
POS_WEIGHT_POWER = 0.5

random.seed(SEED)
torch.manual_seed(SEED)

## Data Split

I split the provided training cache into a training part and a calibration part. The calibration part is used for thresholds and the phrase router. The official validation cache is only used for reporting the final score.

In [ ]:
def project_root() -> Path:
    cwd = Path.cwd()
    return cwd.parent if cwd.name == "notebooks" else cwd


ROOT = project_root()
ARTEFACTS = ROOT / "artefacts"
RESULTS = ARTEFACTS / "results"


def find_cache(split: str) -> Path:
    matches = sorted(ARTEFACTS.glob(f"*legal_dutch*{split}"))
    if not matches:
        raise FileNotFoundError(f"No legal_dutch {split} cache found")
    return matches[0]


def split_train_calibration(length: int, calibration_fraction: float = 0.2):
    generator = torch.Generator().manual_seed(SEED)
    indices = torch.randperm(length, generator=generator).tolist()
    cal_size = round(length * calibration_fraction)
    calibration_idx = sorted(indices[:cal_size])
    train_idx = sorted(indices[cal_size:])
    return train_idx, calibration_idx


if RUN_TRAINING:
    train_cache = VectorCache.load(find_cache("train"))
    valid_cache = VectorCache.load(find_cache("valid"))
    train_idx, calibration_idx = split_train_calibration(len(train_cache))
    train_core = train_cache.dataset.select(train_idx)
    calibration = train_cache.dataset.select(calibration_idx)
    official_valid = valid_cache.dataset
    print(len(train_core), len(calibration), len(official_valid))

## Helper Functions

These helper functions calculate F1, class weights, and per-label thresholds. The important idea is that rare labels receive more attention during training, while thresholds are selected separately for each label on the calibration split.

In [ ]:
def add_onehot(dataset):
    onehot = OneHot(num_classes=N_CLASSES, label_col="labels", target_col="onehot")
    return dataset.map(onehot)


def make_loader(dataset, shuffle: bool):
    collate = Collate(
        embedding_col=EMBEDDING_COL,
        target_col="onehot",
        padder=FixedPadding(max_chunks=MAX_CHUNKS),
    )
    return DataLoader(add_onehot(dataset), batch_size=BATCH_SIZE, shuffle=shuffle, collate_fn=collate)


def target_matrix(dataset):
    targets = torch.zeros((len(dataset), N_CLASSES), dtype=torch.float32)
    labels_column = dataset.with_format(None)["labels"]
    for row_idx, labels in enumerate(labels_column):
        for label in labels:
            targets[row_idx, int(label)] = 1.0
    return targets


def compute_pos_weight(targets):
    positives = targets.sum(dim=0)
    negatives = targets.shape[0] - positives
    raw = (negatives / positives.clamp(min=1.0)).pow(POS_WEIGHT_POWER)
    weights = raw.clamp(min=1.0, max=POS_WEIGHT_CAP)
    return torch.where(positives > 0, weights, torch.full_like(weights, POS_WEIGHT_CAP))


def f1_score(targets, probabilities, thresholds, average="micro"):
    if not torch.is_tensor(thresholds):
        thresholds = torch.full((1, targets.shape[1]), float(thresholds))
    predictions = (probabilities > thresholds).float()
    tp = predictions * targets
    fp = predictions * (1 - targets)
    fn = (1 - predictions) * targets
    if average == "micro":
        precision = tp.sum() / (tp.sum() + fp.sum() + 1e-7)
        recall = tp.sum() / (tp.sum() + fn.sum() + 1e-7)
        return float((2 * precision * recall / (precision + recall + 1e-7)).item())
    precision = tp.sum(dim=0) / (tp.sum(dim=0) + fp.sum(dim=0) + 1e-7)
    recall = tp.sum(dim=0) / (tp.sum(dim=0) + fn.sum(dim=0) + 1e-7)
    per_label = 2 * precision * recall / (precision + recall + 1e-7)
    return float(per_label.mean().item())


def choose_per_label_thresholds(targets, probabilities):
    grid = [x / 100 for x in range(5, 95, 5)]
    selected = []
    for label in range(targets.shape[1]):
        y = targets[:, [label]]
        p = probabilities[:, [label]]
        if int(y.sum().item()) < 10:
            selected.append(0.25)
            continue
        best_threshold, best_score = 0.5, -1
        for threshold in grid:
            score = f1_score(y, p, threshold, average="micro")
            if score > best_score:
                best_threshold, best_score = threshold, score
        selected.append(best_threshold)
    return torch.tensor(selected).view(1, -1)

## Final Attention Model Training

This is the actual final-model training logic. The model uses attention over document chunks, then a small neural network for the 32 labels. When `RUN_TRAINING=True`, this cell trains the model, predicts on calibration and validation, and reports F1.

In [ ]:
def build_attention_model(hidden_size: int):
    return Serial([AttentionAggregator(hidden_size), NeuralNet(hidden_size, N_CLASSES)])


@torch.no_grad()
def predict(model, dataset, device):
    model.eval()
    probabilities, targets = [], []
    for x, y in make_loader(dataset, shuffle=False):
        logits = model(x.to(device))
        probabilities.append(torch.sigmoid(logits).cpu())
        targets.append(y.cpu())
    return torch.cat(probabilities), torch.cat(targets)


def train_attention_model(train_dataset, calibration_dataset, valid_dataset):
    hidden_size = train_dataset[0][EMBEDDING_COL].shape[-1]
    device = detect_device()
    model = build_attention_model(hidden_size).to(device)
    train_targets = target_matrix(train_dataset)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=compute_pos_weight(train_targets).to(device))
    optimizer = optim.Adam(model.parameters(), lr=LR)

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0.0
        for x, y in make_loader(train_dataset, shuffle=True):
            optimizer.zero_grad()
            logits = model(x.to(device))
            loss = loss_fn(logits, y.to(device))
            loss.backward()
            optimizer.step()
            total_loss += float(loss.item())
        print(f"epoch {epoch + 1}: loss={total_loss:.3f}")

    calibration_probs, calibration_targets = predict(model, calibration_dataset, device)
    valid_probs, valid_targets = predict(model, valid_dataset, device)
    return calibration_probs, calibration_targets, valid_probs, valid_targets


if RUN_TRAINING:
    cal_probs, cal_targets, val_probs, val_targets = train_attention_model(
        train_core, calibration, official_valid
    )
    thresholds = choose_per_label_thresholds(cal_targets, cal_probs)
    print("attention micro-F1", f1_score(val_targets, val_probs, thresholds, "micro"))
    print("attention macro-F1", f1_score(val_targets, val_probs, thresholds, "macro"))

## Legal Phrase Router

The final small correction handles a specific legal confusion pattern. Labels 28, 29, and 31 are all related to `art. 7:3 BW`, but label 28 is linked to WVG/voorkeursrecht and label 31 is linked to Omgevingswet/9.9. The router only adjusts those closely related labels.

In [ ]:
def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", text.lower().replace("?", "e"))


def phrase_features(text: str):
    text = normalize_text(text)
    return {
        "article_7_3": bool(re.search(r"art(?:ikel|\.)?\s*7\s*[:.]\s*3", text)),
        "wvg": bool(re.search(r"wvg|wet voorkeursrecht gemeenten|voorkeursrecht", text)),
        "omgevingswet_9_9": bool(re.search(r"omgevingswet|9\s*\.\s*9", text)),
    }


def apply_phrase_router(probabilities, texts, router_boost=0.25, router_suppression=0.15):
    corrected = probabilities.clone()
    for i, text in enumerate(texts):
        features = phrase_features(text)
        if not features["article_7_3"]:
            continue
        if features["wvg"] and not features["omgevingswet_9_9"]:
            corrected[i, 28] += router_boost
            corrected[i, 31] -= router_suppression
            corrected[i, 29] -= router_suppression * 0.5
        elif features["omgevingswet_9_9"]:
            corrected[i, 31] += router_boost
            corrected[i, 28] -= router_suppression
            corrected[i, 29] -= router_suppression * 0.5
        else:
            corrected[i, 29] += router_boost * 0.5
            corrected[i, 28] -= router_suppression * 0.5
            corrected[i, 31] -= router_suppression * 0.5
    return corrected.clamp(0.0, 1.0)


if RUN_TRAINING:
    cal_texts = list(calibration.with_format(None)["text"])
    val_texts = list(official_valid.with_format(None)["text"])
    cal_corrected = apply_phrase_router(cal_probs, cal_texts)
    val_corrected = apply_phrase_router(val_probs, val_texts)
    thresholds = choose_per_label_thresholds(cal_targets, cal_corrected)
    print("final micro-F1", f1_score(val_targets, val_corrected, thresholds, "micro"))
    print("final macro-F1", f1_score(val_targets, val_corrected, thresholds, "macro"))

## Saved Experiment Results

The table below summarizes the completed experiment results. This lets the notebook be reviewed quickly without rerunning training every time.

In [ ]:
result_files = {
    "stage1": RESULTS / "stage1_results.csv",
    "reranker": RESULTS / "reranker_results.csv",
    "label_graph": RESULTS / "label_graph_results.csv",
    "macro": RESULTS / "macro_results.csv",
    "targeted": RESULTS / "targeted_results.csv",
}

frames = []
for name, path in result_files.items():
    if path.exists():
        frame = pd.read_csv(path)
        frame["source"] = name
        frames.append(frame)

results = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
for column in ["official_valid_micro_f1", "official_valid_macro_f1"]:
    if column in results:
        results[column] = pd.to_numeric(results[column], errors="coerce")

model_order = [
    ("Masked mean baseline", "masked_mean_mlp", "official_valid_micro_f1"),
    ("MoE", "moe_mlp", "official_valid_micro_f1"),
    ("Attention", "attention_mlp", "official_valid_micro_f1"),
    ("Reranker", "legal_evidence_reranker", "official_valid_micro_f1"),
    ("Label graph", "label_graph_corrected_attention", "official_valid_micro_f1"),
    ("Macro-focused attention", "rare_label_boosted_attention", "official_valid_macro_f1"),
    ("Final phrase router", "rare_legal_phrase_router", "official_valid_macro_f1"),
]

rows = []
for display_name, model_name, sort_col in model_order:
    subset = results[results["model_name"] == model_name] if not results.empty else pd.DataFrame()
    if subset.empty:
        continue
    row = subset.sort_values(sort_col, ascending=False).iloc[0]
    rows.append({
        "Stage": display_name,
        "Validation micro-F1": row["official_valid_micro_f1"],
        "Validation macro-F1": row["official_valid_macro_f1"],
    })

summary = pd.DataFrame(rows)
summary

In [ ]:
if not summary.empty:
    ax = summary.set_index("Stage").plot.bar(figsize=(10, 5), rot=30)
    ax.set_title("Validation F1 by experiment stage")
    ax.set_ylabel("F1")
    ax.set_xlabel("")
    ax.legend(loc="lower right")
    plt.tight_layout()

## Router Label Changes

In [ ]:
label_changes = pd.DataFrame([
    {"Label": 8, "Legal fact": "koopovereenkomst (beeindiging)", "Before router F1": 0.136, "After router F1": 0.136},
    {"Label": 23, "Legal fact": "vervallen verklaring beperkt zakelijk recht", "Before router F1": 0.000, "After router F1": 0.000},
    {"Label": 26, "Legal fact": "overdracht om niet", "Before router F1": 0.000, "After router F1": 0.000},
    {"Label": 28, "Legal fact": "koopovereenkomst, art. 7:3 BW en 10 WVG", "Before router F1": 0.500, "After router F1": 0.857},
    {"Label": 29, "Legal fact": "koopovereenkomst, art. 7:3 BW", "Before router F1": 0.578, "After router F1": 0.622},
    {"Label": 31, "Legal fact": "koopovereenkomst, art. 7:3 BW en 9.9 Omgevingswet", "Before router F1": 0.417, "After router F1": 0.842},
])
label_changes

## Final Conclusion

The final validation result is **0.8647 micro-F1** and **0.6503 macro-F1**. The main improvement came from focusing on macro-F1, handling label imbalance, tuning thresholds per label, and using a small amount of legal knowledge for the most confusing `art. 7:3` labels. The approach is not perfect, especially for very rare labels, but the final model improves both overall and label-balanced performance.